### Read Files

In [1]:
using PowerModels
using JuMP,Gurobi, DataFrames, XLSX,Random,Distributions
# Specify the path to your MATPOWER file
file_path = "/Users/Patron/Documents/Switch_Instances/case118Blumsack.m"
file_path_excel = "/Users/Patron/Documents/Switch_Instances/RandomDemands.xlsx"

# Parse the MATPOWER case file
network = PowerModels.parse_file(file_path);

# Read the Excel data containing random parameters 
param_data = XLSX.readtable(file_path_excel, "118_15") |> DataFrame

center_data = [56 27 42 50 0 59 29 40 0 0 80 56 46 17 99 37 24 69 56 25 16 23 7 20 0 0 83 31 28 0 47 65 29 71 35 32 0 0 39 74 48 111 31 30 65 40 42 35 88 24 27 28 24 124 70 88 22 25 278 92 0 83 0 0 0 47 41 0 0 75 0 20 21 75 47 72 76 79 46 130 0 65 33 23 24 32 0 63 0 450 19 79 26 45 44 42 17 42 57 38 24 10 38 51 33 46 51 6 13 51 0 73 13 18 23 199 22 42];
Pd = 0.01.*collect(center_data[1,:])
theta = rand(Uniform(0, 0.2), length(Pd))
#theta = 1
Pdstdev = theta .* Pd;

[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 1 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 1 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 54 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 54 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 101 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 101 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 41 from -360.0 to -60.0 deg

### Cost data

In [2]:
G = network["gen"]
c = zeros(length(network["bus"])) #Linear Cost coefficient with zero intercept
for (gen_id,generator) in G
     c[generator["gen_bus"]] = generator["cost"][1]    #generator linear cost funcion slope with 0 intercept
end

### Parameters

In [13]:
function parameters(network)
    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Parameters
    B_ij = zeros(length(L)) # Suspectance valuesL
    fbar_ij = zeros(length(L)) # Upper bound on the load
    p_max = zeros(length(B)) #Max power ouput
    p_min = zeros(length(B)) #Min power output

    for (branch_id,branch) in L
        B_ij[parse(Int,branch_id)] = -branch["br_x"]/(branch["br_r"]^2 + branch["br_x"]^2)  # B_ij = -reactance/(resistance^2+reactance^2)
        fbar_ij[parse(Int,branch_id)] = branch["rate_a"]   #f_max = rate A
    end

    for (gen_id,generator) in G
        p_max[generator["gen_bus"]] = generator["pmax"]     # max power 
        p_min[generator["gen_bus"]] = generator["pmin"]     # min power
    end

    return B_ij,fbar_ij,p_max,p_min
end
_,_,a,b= parameters(network)
a

118-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 5.5
 ⋮
 0.0
 1.36
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

### Offline calculations

In [4]:
function offline_calc(network, Pd, cost)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    Penalty = 10000

    nb = length(B)    # Number of buses
    
    B_ij,f_ij,p_max,p_min = parameters(network)

    # Objective
    @objective(model, Min,
        sum(cost[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )
    
    # Constraints

    # 1) generation limits
    @constraint(model, [i=1:length(B)], pg[i] >= p_min[i])
    @constraint(model, [i=1:length(B)], pg[i] <= p_max[i])

    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - Pd[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end
    #@constraint(model, [i = 1:length(L)], x[i] <= 1)
    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    
   
    optimize!(model)
    

    if termination_status(model) == MOI.OPTIMAL
        result = value.(x)
        return objective_value(model),collect(value.(x)), collect(value.(pg))
    else
        return -1,[],[]
    end
end

offline_calc (generic function with 1 method)

### Generate demand scenarios and random cost vector

In [5]:
function generate_scenarios(Pd,numscenarios)
    samples = zeros(numscenarios,length(Pd))
    # Global min
    #global_min = minimum(Pd)
    #global_max = maximum(Pd)
    
    for scenario in 1:numscenarios
        #global_value = round(rand(Uniform(global_min, global_max)),digits=0)  # Global value
        global_factor = rand(Uniform(0.8, 1.2)) #Global factor
        for i in 1:length(Pd)
            center = Pd[i]
            if center == 0
                #min_val = 0
                #max_val = 1  # Default range for zero-center buses
                local_factor_i = rand(Uniform(0,1))
            else
            #min_val = max(0, center * 0.80)  # 20% below center
            #max_val = center * 1.2         # 20% above center
                local_factor_i = 1 + rand(Uniform(-0.1, 0.1))  # ±20% local
            end
            bus_demand_i = Pd[i] * 1 * local_factor_i
            samples[scenario,i] = bus_demand_i
            #samples[scenario,i]= round(rand(Uniform(min_val, max_val))*(1+(global_value*0.0)/global_max),digits=0)
        end
    end
    return samples
end
function generate_normal_scenarios(Pd,numscenarios,Pdstdev)
    samples = zeros(numscenarios,length(Pd))
    for scenario in 1:numscenarios
        globaldem = max(rand(Normal(0.2,0.05)),0)
        for i in 1:length(Pd)
            samples[scenario,i]=max(rand(Normal(Pd[i],Pdstdev[i])),0)
        end
    end
    return samples
end


function random_cost(c)
    cost = zeros(length(c))
    for i in 1:length(c)
        variation = rand(Uniform(0.8,1.2))
        cost[i] = variation*c[i]
    end
    return cost
end

random_cost (generic function with 1 method)

### Sampling

In [63]:
function sampling_for_offline_calc(network,Pd, c,total_scenarios; capacity_factor::Real = 2.0 )
    # 1) unpack network parameters
    B_ij,fbar_ij,p_max_base,p_min_base = parameters(network)

    # index sets
    buses    = sort(parse.(Int, collect(keys(network["bus"]))))
    branches = sort(parse.(Int, collect(keys(network["branch"]))))
    nb       = length(buses)
    nl       = length(branches)

    # 2) pre-allocate storage
    demand_samples = zeros(Float64,total_scenarios, nb)
    x_solutions    = zeros(total_scenarios, nl)
    pg_solutions   = zeros(total_scenarios, nb  )

    # 3) sampling loop
    found = 0
    while found < total_scenarios
        # generate one draw of demand & cost
        ds = generate_normal_scenarios(Pd, 1,Pdstdev)

        # make *local* scaled copies of your capacities
        p_max = p_max_base .* capacity_factor
        p_min = p_min_base .* capacity_factor

        # random cost vector
        cost = random_cost(c)

        # try to solve
        obj,x,pg = offline_calc(network,generate_scenarios(Pd, 1), cost)

        # if we get here, it solved to optimality
        if obj!=-1
            found += 1
            demand_samples[found, :] = ds
            x_solutions[found, :] = x
            pg_solutions[found, :] = pg

        else
            println("Scenario is infeasible, trying again")
            # just loop and sample again
        end
    end

    return demand_samples, x_solutions, pg_solutions
end
demand_samples,x_solutions, pg_solutions = sampling_for_offline_calc(network,Pd,c,5)

([0.45353789945987516 0.2688603045808017 … 0.21753625745187216 0.4052447214604339; 0.6461185860095339 0.2704629853616687 … 0.24145561599213022 0.4360605478030413; … ; 0.7308639237936237 0.2700216444346411 … 0.22221252064132754 0.3480693607626954; 0.6368846703688095 0.2680255877350788 … 0.19463586625269672 0.3674986199567369], [1.0 0.0 … 1.0 0.0; 1.0 1.0 … 1.0 -0.0; … ; 1.0 1.0 … 0.0 1.0; 1.0 1.0 … -0.0 -0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])

### Online samples

In [64]:
online_samples = generate_scenarios(Pd,200);
num_samples = 200
nb          = length(network["bus"])
cost_samples           = zeros(Float64, num_samples, nb)

for s in 1:num_samples
    cost_samples[s, :] = random_cost(c)
end

### Decoupling binary and continuous decision variable

In [65]:
function solve_for_continuous_var(network,sample,c, solved_x)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 300.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    Penalty = 10000

    B_ij,f_ij,p_max,p_min = parameters(network)

    # Objective
    @objective(model, Min,
        sum(c[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )

    # Constraints

    # 1) generation limits, gated by z
    @constraint(model, [i=1:length(B)],
        pg[i] >= p_min[i] 
    )
    @constraint(model, [i=1:length(B)],
        pg[i] <= p_max[i] 
    )

    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - sample[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end

    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    # 4) forcing binary variable to take a fixed value
    @constraint(model, x == solved_x )
    
    optimize!(model)
    # Check solver status
    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),collect(value.(pg))
    else
        return -1,[]
    end
end 

function Calculations_For_Extensive_Form(network,online_samples,cost_samples,x_chosen)
    total_scenarios = size(online_samples,1)
    total_solutions = size(x_chosen,1)
    obj_value = zeros(total_scenarios,total_solutions)
    for i in 1:total_scenarios
        sample = online_samples[i,:]
        cost = cost_samples[i,:]
        for j in 1:total_solutions
            obj_value[i,j], _ = solve_for_continuous_var(network,sample,cost, x_chosen[j,:])
        end
    end
    return obj_value
end


Calculations_For_Extensive_Form (generic function with 1 method)

In [66]:
obj_matrix = Calculations_For_Extensive_Form(network,online_samples,cost_samples,x_solutions)

200×5 Matrix{Float64}:
 3070.41  2998.14  2990.76  3202.04  3035.49
 3169.04  3210.25  3221.61  3267.17  3209.58
 3612.16  3771.8   3825.59  3686.28  3597.01
 3050.91  3004.23  3008.44  3164.58  3134.25
 2666.24  2742.5   2725.32  2840.43  2701.35
 3377.11  3587.95  3607.85  3439.9   3333.21
 3743.14  3631.23  3621.78  3818.38  4038.27
 3386.28  3325.7   3324.67  3545.12  3488.22
 3266.61  3353.33  3370.22  3360.14  3291.2
 2656.63  2727.1   2713.36  2774.35  2692.72
    ⋮                                
 2680.12  2667.17  2665.71  2802.63  2655.83
 3053.92  3002.79  3003.17  3200.86  3120.89
 4442.0   4584.83  4576.43  3504.18  4527.48
 3951.98  3810.69  3806.5   3543.03  4202.04
 3139.53  3189.98  3203.22  3321.53  3226.27
 3134.85  3247.59  3207.43  3259.69  3182.58
 2903.17  2862.64  2869.17  3060.6   2975.97
 3060.27  3077.18  3065.33  3254.46  3134.94
 4436.92  4627.65  4627.11  3643.86  4386.17

### Reformulated Extensive Form

In [76]:
function Reformulated_Extensive_Form(S,l,K,obj_value)
    # Initialize model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Total scenarios

    # Decision Variables
    @variable(model, z[1:l], Bin)
    @variable(model, w[1:l , 1:S], Bin)

    # Constraints
    for scenario in 1:S
        for solution in 1:l
            if obj_value[scenario,solution] == -1
                @constraint(model, w[solution,scenario] == 0)   ## weights of infeasible scenario/first stage decision pair is 0
            end
        end
    end
            
        
    for j in 1:S
        @constraint(model, [i in 1:l], w[i,j] <= z[i])
        @constraint(model, sum(w[i,j] for i in 1:l) == 1)
    end
    @constraint(model, sum(z[i] for i in 1:l) == K)

    # Objective
    objective = 0.0
    for scenario in 1:S
        for solution in 1:l
            objective += w[solution,scenario]*obj_value[scenario,solution]
        end
    end

    # Define the objective function
    @objective(model, Min, objective/S)

    # Solve the model
    #set_silent(model)
    optimize!(model)
    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(z)
    else
        return -1,[]
    end    
end 
Solution,z=Reformulated_Extensive_Form(200,5,5,obj_matrix)

(3106.0759321771834, [1.0, 1.0, 1.0, 1.0, 1.0])

### Test Sample

In [70]:
test_sample = generate_scenarios(Pd,5);
num_samples = 10
nb          = length(network["bus"])
cost_test           = zeros(Float64, num_samples, nb)

for s in 1:num_samples
    cost_test[s, :] = random_cost(c)
end

### Offline Calculation on Cost Data

In [71]:
function OfflineTest(network,test_sample,cost_test)
    num_rows = size(test_sample, 1)
    offline_cost= zeros(num_rows)

    #Offline calculations for the test data
    total_cost = 0.0
    count = 0
    for i in 1:num_rows
        obj_value,_,_ = offline_calc(network,test_sample[i,:],cost_test[i,:])
        offline_cost[i] = obj_value
        if obj_value != -1
            total_cost += obj_value
            count += 1
        end
    end
    total_offline_cost = total_cost/count
    return total_offline_cost,offline_cost
end
original_cost,offline_cost_list = OfflineTest(network,test_sample,cost_test);

### Second Stage Evaluation

In [72]:
function Second_Stage_Cost(network,sample,c_sample, x_chosen,z)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    @variable(model, w[1:length(z)], Bin)
    Penalty = 10000

    nb = length(B)    # Number of buses
    
    B_ij,f_ij,p_max,p_min = parameters(network)

    # Objective
    @objective(model, Min,
        sum(c_sample[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )
    
    # Constraints

    # 1) generation limits, gated by z_gen
    @constraint(model, [i=1:length(B)],
        pg[i] >= p_min[i]
    )
    @constraint(model, [i=1:length(B)],
        pg[i] <= p_max[i]
    )
    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - Pd[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end
    #@constraint(model, [i = 1:length(L)], x[i] <= 1)
    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    @constraint(model, [i in 1:length(z)], w[i] <= z[i])
    @constraint(model, sum(w[i] for i in 1:length(z)) == 1)
    @constraint(model, x == sum(w[i] * x_chosen[i,:] for i in 1:length(z)))
   
    optimize!(model)

    # Solve the model
    set_silent(model)
    optimize!(model)

    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(w)
    else
        #println("Solver did not find an optimal solution.")
        return -1,[]
    end
end  

Second_Stage_Cost (generic function with 1 method)

### Evaluation

In [77]:
function Evaluation(network,test_sample,cost_test, offline_cost_list,x_chosen,z)
    num_rows = size(test_sample, 1)
    online_list = zeros(num_rows)
    ### Second stage cost
    model_cost = 0.0
    count = 0
    for index in 1:num_rows
        if offline_cost_list[index] != -1
            model_obj_value,_ = Second_Stage_Cost(network,test_sample[index,:],cost_test[index,:],x_chosen,z)
            online_list[index] = model_obj_value
            if model_obj_value != -1
                model_cost += model_obj_value
                count += 1
            end
        else
            online_list[index] = -1
        end
    end
    total_model_cost = model_cost / count

    return total_model_cost, online_list
end 
model_cost, model_cost_list = Evaluation(network,test_sample,cost_test,offline_cost_list,x_solutions,z);

In [74]:
original_cost

2916.105872259983

In [78]:
model_cost

3013.568863168908